In [36]:
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb
!apt-get -f install -y
!google-chrome --version


--2026-01-19 22:08:33--  https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
Resolving dl.google.com (dl.google.com)... 74.125.201.93, 74.125.201.136, 74.125.201.91, ...
Connecting to dl.google.com (dl.google.com)|74.125.201.93|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 119196224 (114M) [application/x-debian-package]
Saving to: ‘google-chrome-stable_current_amd64.deb.1’

google-chrome-stabl 100%[===================>] 113.67M  81.4MB/s    in 1.4s    

2026-01-19 22:08:34 (81.4 MB/s) - ‘google-chrome-stable_current_amd64.deb.1’ saved [119196224/119196224]

(Reading database ... 118398 files and directories currently installed.)
Preparing to unpack google-chrome-stable_current_amd64.deb ...
Unpacking google-chrome-stable (144.0.7559.59-1) over (144.0.7559.59-1) ...
Setting up google-chrome-stable (144.0.7559.59-1) ...
Processing triggers for mailcap (3.70+nmu1ubuntu1) ...
Processing triggers for man-db (2.10.2-1) ...
Reading package lis

In [37]:
# Baixa o Chromedriver compatível com a versão do Chrome
!CHROME_VERSION=$(google-chrome --version | grep -oP '\d+')
!wget -O /tmp/chromedriver.zip https://chromedriver.storage.googleapis.com/$CHROME_VERSION/chromedriver_linux64.zip
!unzip /tmp/chromedriver.zip -d /usr/local/bin/
!chmod +x /usr/local/bin/chromedriver
!which chromedriver
!chromedriver --version


--2026-01-19 22:09:05--  https://chromedriver.storage.googleapis.com//chromedriver_linux64.zip
Resolving chromedriver.storage.googleapis.com (chromedriver.storage.googleapis.com)... 173.194.193.207, 173.194.194.207, 173.194.195.207, ...
Connecting to chromedriver.storage.googleapis.com (chromedriver.storage.googleapis.com)|173.194.193.207|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-01-19 22:09:05 ERROR 404: Not Found.

Archive:  /tmp/chromedriver.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /tmp/chromedriver.zip or
        /tmp/chromedriver.zip.zip, and cannot find /tmp/chromedriver.zip.ZIP, period.
chmod: cannot access '/usr/local/bin/chromedriver': No such file or directory
/bin/bash: line 1: 

In [28]:
!which chromium
!which chromedriver


/usr/bin/chromedriver


In [39]:
# =============================
# 0. IMPORTS
# =============================

import time
import csv


from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC






# =============================
# 1. CONFIGURAÇÕES
# =============================
url_classificacao = "https://superscore.live/pt-BR/futebol/competicoes/brasileiro-serie-a/0ybfrxuc/classificacao?season=79lXJHlwqvfEJlUjFl96fs"



chrome_options = Options()
chrome_options.binary_location = "/usr/bin/google-chrome-stable"  # Chrome instalado manualmente
chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--disable-gpu")
chrome_options.add_argument("--window-size=1920,1080")

# Inicializa o driver usando chromedriver compatível
service = Service("/usr/local/bin/chromedriver")


driver = webdriver.Chrome(service=service, options=chrome_options)




wait = WebDriverWait(driver, 30)

# =============================
# 2. ABRE A PÁGINA DE CLASSIFICAÇÃO
# =============================
try:
    print(f"Acessando: {url_classificacao}")
    driver.get(url_classificacao)

    # 2.1 Aceitar banner de cookies
    try:
        cookie_button = wait.until(
            EC.presence_of_element_located((By.ID, "onetrust-accept-btn-handler"))
        )
        driver.execute_script("arguments[0].click();", cookie_button)
        wait.until(EC.invisibility_of_element_located((By.ID, "onetrust-banner-sdk")))
        print("Banner de cookies aceito.")
    except:
        print("Banner de cookies não encontrado ou já aceito.")

    # 2.2 Espera pela tabela de classificação
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div[data-testid='standings-header']")))
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div[data-testid='standings-main-container']")))
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".team-name")))
    print("Tabela de classificação carregada.")

    # =============================
    # 3. COLETA LINKS DOS TIMES
    # =============================
    team_name_elements = driver.find_elements(By.CSS_SELECTOR, ".team-name")
    time_links = list(dict.fromkeys([
        el.find_element(By.XPATH, "./ancestor::a").get_attribute("href")
        for el in team_name_elements
        if el.get_attribute("href")
    ]))
    print(f"Encontrados {len(time_links)} times.")

    dados_jogadores = []

    # =============================
    # 4. ITERAÇÃO POR TIME
    # =============================
    for i, link_time in enumerate(time_links, 1):
        print(f"\nProcessando time {i}/{len(time_links)}: {link_time}")
        driver.get(link_time)

        try:
            esquadrao_tab = wait.until(
                EC.element_to_be_clickable((By.XPATH, "//div[contains(text(),'Esquadrão')]"))
            )
            esquadrao_tab.click()
            wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".player-card a")))
        except:
            print("  Não foi possível carregar o elenco do time.")
            continue

        jogador_links = list(dict.fromkeys([
            el.get_attribute("href")
            for el in driver.find_elements(By.CSS_SELECTOR, ".player-card a")
            if el.get_attribute("href")
        ]))

        # =============================
        # 5. ITERAÇÃO POR JOGADOR
        # =============================
        for link_jogador in jogador_links:
            driver.get(link_jogador)

            try:
                estatisticas_tab = wait.until(
                    EC.element_to_be_clickable((By.XPATH, "//div[contains(text(),'Estatísticas')]"))
                )
                estatisticas_tab.click()
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".stats-section")))
            except:
                print(f"    Estatísticas não disponíveis para {link_jogador}.")
                continue

            try:
                nome = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".player-name"))).text
            except:
                nome = ""

            # Coleta dados gerais
            dados_gerais = {}
            for block in driver.find_elements(By.CSS_SELECTOR, ".player-info .info-block"):
                try:
                    label = block.find_element(By.CSS_SELECTOR, ".label").text.strip()
                    value = block.find_element(By.CSS_SELECTOR, ".value").text.strip()
                    dados_gerais[label] = value
                except:
                    continue

            # Coleta estatísticas
            estatisticas = {}
            for block in driver.find_elements(By.CSS_SELECTOR, ".stats-section .stat-block"):
                try:
                    label = block.find_element(By.CSS_SELECTOR, ".label").text.strip()
                    value = block.find_element(By.CSS_SELECTOR, ".value").text.strip()
                    estatisticas[label] = value
                except:
                    continue

            jogador_data = {"Nome": nome}
            jogador_data.update(dados_gerais)
            jogador_data.update(estatisticas)
            dados_jogadores.append(jogador_data)
            print(f"    Dados coletados: {nome}")

    # =============================
    # 6. SALVA CSV
    # =============================
    if dados_jogadores:
        fieldnames = set().union(*(d.keys() for d in dados_jogadores))
        fieldnames.discard("Nome")
        fieldnames = ["Nome"] + sorted(fieldnames)

        with open("estatisticas_jogadores_brasileirao_2025.csv", "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(dados_jogadores)

        print(f"\nCSV salvo com {len(dados_jogadores)} jogadores.")
    else:
        print("\nNenhum dado coletado.")

finally:
    driver.quit()
    print("Driver fechado com sucesso.")


NoSuchDriverException: Message: Unable to obtain driver for chrome; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors/driver_location
